# 02 · From-scratch DPO

Run this notebook twice, independently from the same SFT best checkpoint: once for HH, once for UltraFeedback.

Upload this notebook to Kaggle. Enable Internet; select **GPU T4 x2** for training/generation. The judge notebook needs benchmark model access, not a GPU. Enter your public GitHub URL below. No GitHub credentials are needed.


## 1. Configuration

Run All defaults to smoke checks only. After inspecting them, set `RUN_FULL=True`. Use a new run name if changing experimental settings; completed runs are skipped.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/0x0shephard/genai-pa2.git"
GIT_REF = "main"  # After the first run, use its printed commit SHA for all final runs.
REPO_DIR = Path("/kaggle/working/pa2-repo")
OUTPUT_ROOT = Path("/kaggle/working/outputs")
PREPARED_ROOT = Path("/kaggle/working/prepared")
RUN_SMOKE = True
RUN_FULL = False  # Inspect smoke output, then set True and rerun the configuration/full cells.

TRACK = "hh"  # Change to "ultrafeedback" for the second independent run.
SFT_BEST = Path("/kaggle/input/YOUR-SFT-OUTPUT/outputs/sft/best")
RUN_NAME = f"dpo_{TRACK}"
RESUME_STATE = None  # Saved outputs/.../resume.pt, with its sibling best directory.


## 2. Clone the chosen GitHub revision and install dependencies


In [ ]:
assert "YOUR_" not in REPO_URL, "Enter your public GitHub repository URL in the configuration cell."
assert REPO_URL.startswith("https://github.com/"), "Use a public GitHub HTTPS URL."
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    assert origin.rstrip("/").removesuffix(".git") == REPO_URL.rstrip("/").removesuffix(".git"), "Existing checkout belongs to another repository."
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip(), "Checkout has local changes; preserve them before changing revisions."
subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
os.environ["ALIGNMENT_GIT_REF"] = GIT_REF
os.environ["PYTHONPATH"] = str(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/pa2-hf-cache"  # Caches are disposable, not saved notebook artifacts.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
print("Git commit:", COMMIT)

def run(*args):
    subprocess.check_call([sys.executable, "-m", "alignment", *map(str, args)], cwd=REPO_DIR)

def need(path, label):
    path = Path(path)
    if not path.exists():
        print("Available attachments:", list(Path("/kaggle/input").glob("*/*")))
        raise FileNotFoundError(f"Attach the saved {label} output and update its configuration path: {path}")
    return path

def completed(path):
    summary = Path(path) / "summary.json"
    return summary.exists() and json.loads(summary.read_text()).get("completed", False)

def show_samples(path, count=5):
    for row in json.loads(Path(path).read_text())[:count]:
        print("PROMPT:", row["instruction"])
        print("RESPONSE:", row["output"], "\n")


In [ ]:
# Keep Kaggle's installed CUDA PyTorch. These requirements do not pin torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt"), "--upgrade-strategy", "only-if-needed"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"])
subprocess.check_call(["nvidia-smi"])
subprocess.check_call([sys.executable, "-c", "import torch, transformers, datasets; print('torch', torch.__version__, 'transformers', transformers.__version__, 'datasets', datasets.__version__); print('GPU count:', torch.cuda.device_count()); assert torch.cuda.device_count() >= 2, 'Enable GPU T4 x2'"])
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], cwd=REPO_DIR)


## 3. Verify SFT input and run a pilot


In [ ]:
assert TRACK in ("hh", "ultrafeedback")
need(SFT_BEST / "alignment_manifest.json", "SFT best checkpoint")
if RUN_SMOKE:
    smoke_data = PREPARED_ROOT / "smoke"
    smoke_run = OUTPUT_ROOT / f"smoke_dpo_{TRACK}"
    run("prepare", "--track", TRACK, "--output", smoke_data, "--train-size", 64, "--val-size", 8)
    if not completed(smoke_run):
        run("dpo", "--data", smoke_data / TRACK, "--model", SFT_BEST, "--output", smoke_run,
            "--max-steps", 2, "--eval-steps", 1)
    print((smoke_run / "metrics.jsonl").read_text())


## 4. Full independent DPO run

15k/1k pairs, beta 0.1, fixed LR 1e-6, batch 1 × accumulation 32, one epoch. Policy uses GPU 0; frozen SFT reference uses GPU 1.


In [ ]:
if RUN_FULL:
    run("prepare", "--track", TRACK, "--output", PREPARED_ROOT)
    # Keep smoke diagnostics, but release its large duplicate model/optimizer files.
    if RUN_SMOKE:
        for bulky in ("best", "final", "resume.pt", "trainer"):
            path = smoke_run / bulky
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
    destination = OUTPUT_ROOT / RUN_NAME
    if not completed(destination):
        args = ["dpo", "--data", PREPARED_ROOT / TRACK, "--model", SFT_BEST, "--output", destination]
        if RESUME_STATE:
            args += ["--resume", need(RESUME_STATE, "DPO resume state")]
        run(*args)
    run("plot", "--run", destination, "--output", destination / "training.png", "--name", RUN_NAME)
else:
    print("Smoke only. Set RUN_FULL=True after inspecting the pilot.")


## Preserve outputs

Use **Save Version → Save & Run All** after setting `RUN_FULL=True`, or save completed interactive outputs with the appropriate Kaggle version option. Attach that saved notebook's output to the next stage using **Add Input → Notebook Output**. Confirm the files exist in the saved version before ending the session.

New artifacts are under `/kaggle/working/outputs` and `/kaggle/working/prepared`. Model files are large: do not duplicate them into zip archives unless disk permits. For interruption recovery, attach the prior saved output and configure its resume path. A file in a temporary session is preserved only when Kaggle saves the output.
